# OlmOCR benchmark — Kaggle / Colab T4

Run [Allen AI's OlmOCR](https://github.com/allenai/olmocr) (Qwen2-VL-based) against the
[`pdf-plaintext-extraction`](https://github.com/EvanOchsner/pdf-plaintext-extraction) synthetic
gold-set corpus (100 sources × 7 variants = 700 PDFs) on a free CUDA GPU.

**Why this notebook exists.** OlmOCR's inference path requires CUDA (vLLM); my local machine
is Apple-Silicon. Rather than maintain a CUDA-or-MPS-fork wrapper inside the benchmark harness,
the heavy GPU work runs once here as a one-shot batch, and a small Python importer converts the
output into the same row schema (`extractor: "olmocr"`) the rest of the benchmark uses. The
JSONL artifact produced at the end of this notebook merges directly into the local benchmark
results.

**Where to run.** Recommended: Kaggle (Settings → Accelerator → GPU T4 × 1). Colab T4 works
too. Anywhere with CUDA ≥ 11.8 and ~15 GB free VRAM.

**Run mode.** Designed for Kaggle's *Save & Run All* (Commit) — fully detached, outputs
captured with the notebook commit. Expected wall time: 30–90 min.

## Phase 0 — Verify GPU + Python environment

In [ ]:
import subprocess, sys, platform
import torch
assert torch.cuda.is_available(), \
    "Need CUDA GPU. On Kaggle: Settings → Accelerator → GPU T4 × 1."
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Phase 1 — Clone repo + materialize the 700-PDF corpus

The corpus is fully reproducible from the committed `pdf_plaintext_extraction/data/sources/*.txt`
files. `ensure_corpus()` renders the 100 clean PDFs and applies the 6 poisoning techniques to
each (=700 total), then writes a ground-truth manifest. Idempotent: re-running this cell after
a kernel restart will skip every PDF whose SHA-256 already matches.

In [ ]:
import os, pathlib, subprocess
REPO_URL = "https://github.com/EvanOchsner/pdf-plaintext-extraction.git"
REPO_DIR = pathlib.Path("/kaggle/working/pdf-plaintext-extraction") \
    if pathlib.Path("/kaggle").exists() else pathlib.Path.home() / "pdf-plaintext-extraction"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)
subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], check=True)

In [ ]:
# Install our package (and the [benchmark] extras for the importer's score deps).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[benchmark]"],
    check=True,
)

In [ ]:
from pdf_plaintext_extraction.benchmark import ensure_corpus
corpus_root = ensure_corpus()
print("Corpus root:", corpus_root)

# Sanity: exactly 100 clean + 600 poisoned = 700 PDFs.
pdfs = list(corpus_root.rglob("*.pdf"))
print(f"PDFs materialized: {len(pdfs)}  (expect 700)")
assert len(pdfs) == 700, "corpus materialization is incomplete"

## Phase 2 — Install OlmOCR + vLLM

On a T4 (Ampere — no FP8 support), we override the OlmOCR default model from the FP8 release
to a BF16 variant. If a newer BF16 release is available at run time, swap the `MODEL_NAME`
below.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "olmocr"],
    check=True,
)
import olmocr
from olmocr.version import VERSION as OLMOCR_VERSION
print("olmocr version:", OLMOCR_VERSION)

# Default upstream model is allenai/olmOCR-2-7B-1025-FP8 (Hopper only).
# Override for T4 / Ampere compatibility:
MODEL_NAME = "allenai/olmOCR-2-7B-1025"   # bf16 sibling of the FP8 release
print("Model:", MODEL_NAME)

## Phase 3 — Run OlmOCR in batch over all 700 PDFs

The olmocr CLI is workspace-oriented: feed it a list of PDF paths and a writable workspace
directory. It starts a vLLM server, processes each PDF through the VLM, and writes
dolma-doc-shaped JSONL to `<workspace>/results/*.jsonl`.

vLLM amortizes model-load (~30 s) across the whole batch — a single invocation over the full
corpus is *far* faster than per-PDF Python calls.

In [ ]:
import time
WORKSPACE = REPO_DIR / "olmocr_workspace"
WORKSPACE.mkdir(exist_ok=True)

# File-list of every PDF in the corpus (clean + poisoned).
pdf_list_path = WORKSPACE / "pdf_list.txt"
with pdf_list_path.open("w") as fh:
    for p in sorted(corpus_root.rglob("*.pdf")):
        fh.write(str(p) + "\n")
n_pdfs = sum(1 for _ in pdf_list_path.open())
print(f"Queued {n_pdfs} PDFs for OlmOCR.")

t0 = time.perf_counter()
result = subprocess.run(
    [
        sys.executable, "-m", "olmocr", str(WORKSPACE),
        "--pdfs", str(pdf_list_path),
        "--model", MODEL_NAME,
        "--workers", "1",
        "--markdown",
    ],
    check=True,
)
elapsed = time.perf_counter() - t0
print(f"OlmOCR wall time: {elapsed:.1f}s ({elapsed/60:.1f}min, ~{elapsed/n_pdfs:.1f}s/PDF)")

## Phase 4 — Convert OlmOCR output to benchmark-row JSONL

`import_olmocr_workspace()` walks `<workspace>/results/*.jsonl`, matches each result to a
`(source_id, variant)` cell by parsing the PDF path, scores against the ground-truth manifest,
and emits rows with `extractor: "olmocr"` in the same schema as the rest of the benchmark.

In [ ]:
from pdf_plaintext_extraction.benchmark.olmocr_importer import import_olmocr_workspace
OUT_JSONL = pathlib.Path("/kaggle/working/olmocr_rows.jsonl") \
    if pathlib.Path("/kaggle").exists() else REPO_DIR / "olmocr_rows.jsonl"

rows = import_olmocr_workspace(
    workspace=WORKSPACE,
    corpus_root=corpus_root,
    out_jsonl=OUT_JSONL,
    strict=False,  # warn-and-continue on any missing cell
)
print(f"Wrote {len(rows)} rows -> {OUT_JSONL}")
print(f"  unique cells: {len({(r['source_id'], r['variant']) for r in rows})}")
print(f"  errored rows: {sum(1 for r in rows if r['error'])}")

## Phase 5 — Inline summary table

Quick mean-F1-per-variant view so we can verify the run looks sensible before the artifact
leaves the notebook. The clean column is the harness control — should be ≥ 0.95.

In [ ]:
from collections import defaultdict
by_variant = defaultdict(list)
wall_total = 0.0
wall_n = 0
for r in rows:
    if r.get("error"):
        continue
    by_variant[r["variant"]].append(r["f1"])
    if r.get("wall_seconds") is not None:
        wall_total += float(r["wall_seconds"])
        wall_n += 1

print(f"{'variant':16s}{'n':>5s}{'mean F1':>10s}")
print("-" * 31)
for v, fs in sorted(by_variant.items()):
    print(f"{v:16s}{len(fs):>5d}{sum(fs)/len(fs):>10.3f}")
if wall_n:
    print(f"\nMean per-PDF wall_seconds (over {wall_n} rows that recorded it): "
          f"{wall_total/wall_n:.2f}s")
print(f"Total batch wall time (Phase 3 measurement): {elapsed:.1f}s")

## Phase 6 — Save artifact for local pickup

On Kaggle, anything written to `/kaggle/working/` is captured with the notebook commit and
downloadable via the Kaggle web UI or `kaggle kernels output <user>/<notebook>`. On Colab,
download from the file panel.

After downloading locally:

```sh
# Append to the existing benchmark JSONL — run_synthetic --resume keys on
# (extractor, source_id, variant), so the 700 olmocr rows merge cleanly.
cat olmocr_rows.jsonl >> experiments/results/synthetic_20260519T050436.jsonl
```

In [ ]:
import os
print("Artifact path:", OUT_JSONL, "(" + f"{OUT_JSONL.stat().st_size:,} bytes)")
print("Lines:", sum(1 for _ in OUT_JSONL.open()))
print()
print("=== reproducibility marker ===")
print(f"olmocr  : {OLMOCR_VERSION}")
print(f"model   : {MODEL_NAME}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"corpus  : {len(pdfs)} PDFs")
print(f"wall    : {elapsed:.1f}s batch ({elapsed/60:.1f} min)")